**Code Explanation with Notes**

Creating a Spark Session:

    We begin by creating a Spark session to run the PySpark operations.

Generating a DataFrame:

    Using spark.range(10) creates a DataFrame with 10 rows and a single column(id) 
    with numbers ranging from 0 to 9.

    Two additional columns are added:
today: Contains the current date using current_date().

now: Contains the current timestamp using current_timestamp().

Date Manipulation Functions:

    date_add: Adds a specified number of days to the date.
    date_sub: Subtracts a specified number of days from the date.
    datediff: Returns the difference in days between two dates.
    months_between: Returns the number of months between two dates.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

dateDF = spark.range(10).withColumn("Today",current_date()).withColumn("Now", current_timestamp())
dateDF.show(truncate=False)

**date_add and date_sub:**

    date_sub(col("today"), 5): Subtracts 5 days from the current date, so if today is 2026-10-12,
    it returns 2026-10-07.
    date_add(col("today"), 5): Adds 5 days to the current date, returning 2026-10-17.

In [0]:
dateDF.select(
    date_sub(col("Today"),7).alias("Date_Substraction"),
    date_add(col("Today"),7).alias("Date_Addition")
    ).show(1)

**datediff**

    datediff(col("week_ago"), col("today")): Calculates the difference in days between the current 
    date and 7 days ago

In [0]:
dateDF.withColumn("week_ago", date_sub("Today", 7))\
    .select(datediff(col("Today"),col("week_ago")).alias("Date_Diffrence")).show(1)

**months_between**

    months_between(to_date(lit("2016-01-01")), to_date(lit("2017-01-01")): Calculates
    the number of months between January 1, 2016, and January 1, 2017, which is -12
    months because start_date is earlier than end_date.

In [0]:
dateDF.select(
    to_date(lit("2026-08-02")).alias("StartDate"),
    to_date(lit("2025-08-02")).alias("EndDate")
).select(months_between(col("EndDate"),col("StartDate")).alias("DiffBetweenMonths")).show(1)

**Default Date Parsing (to_date):**

    When using to_date(), the default date format is yyyy-MM-dd.
    If the format of 

In [0]:
dateDF.select(
    expr("try_cast('2026-20-12' as date)").alias("IncorrectDate"),
    to_date(lit("2026-08-03")).alias("CorrectDate")
).show(1)

**Handling Custom Date Formats:**

    You can specify a custom date format using the to_date function by providing a
    format string, such as yyyy-dd-MM.
    
    This allows PySpark to correctly parse the dates that deviate from the default
    format.

In [0]:
from pyspark.sql.functions import *

dateformat = "yyyy-dd-MM"
newdataDF = spark.range(1).select(
    to_date(lit("2026-12-03"), dateformat).alias("CorrectDate"),
    to_date(lit("2026-08-02"), dateformat).alias("InCorrectDate")
)
newdataDF.show(1)

**Handling Timestamps:**

    You can use to_timestamp to convert strings with both date and time into a
    timestamp format. This is useful when working with datetime values.
    
    After casting to a timestamp, you can extract various dat

In [0]:
from pyspark.sql.functions import *
newdataDF.select(
    to_timestamp(col("CorrectDate"),dateformat).alias("TimeStamp"),
    year(to_timestamp(col("CorrectDate"),dateformat)).alias("Year"),
    month(to_timestamp(col("CorrectDate"),dateformat)).alias("Month"),
    dayofmonth(to_timestamp(col("CorrectDate"),dateformat)).alias("Day"),
    hour(to_timestamp(col("CorrectDate"),dateformat)).alias("Hour"),
    minute(to_timestamp(col("CorrectDate"),dateformat)).alias("Minute"),
    second(to_timestamp(col("CorrectDate"),dateformat)).alias("Secounds")
).show()

Detailed Explanation of Each Function

1. to_date:
   - Converts a string column to a date column based on the given format. If the format does not match, null is returned.

2. to_timestamp:
   - Converts a string column with date and time information into a timestamp, which includes both date and time.

3. Extracting Date Components:
   - year: Extracts the year from a date or timestamp.
   - month: Extracts the month from a date or timestamp.
   - dayofmonth: Extracts the day of the month from a date or timestamp.
   - hour: Extracts the hour from a timestamp.
   - minute: Extracts the minute from a timestamp.
   - second: Extracts the second from a timestamp.

Sample Output

For the input "2017-11-12" (with the format yyyy-dd-MM), you can expect the following results:
- Year: 2017
- Month: 12
- Day: 11
- Hour: 0 (since no time is provided)
- Minute: 0
- Second: 0

For invalid date strings (like "2017-20-12"), you will get null in the resulting DataFrame.